In [1]:
import os
import time
import json
from pathlib import Path

from qdrant import VectorDB
from qdrant_client import models
from utils import load_config
from dotenv import load_dotenv
from batches import *
from concurrent.futures import ThreadPoolExecutor, as_completed

from google import genai
from google.genai import types
from openai import OpenAI

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

In [4]:
config = load_config("config.ini")

# Connect to QDRANT
qdrant_client = VectorDB(
    port=config.getint("QDRANT", "port"),
    host=config.get("QDRANT", "host")
)

[2026-02-25 14:34:24] Connected to http://localhost:6333/dashboard
[2026-02-25 14:34:24] Storage will be at /Users/marlonselvi/Documents/Programming/tuutrag-open/tuutrag/data/qdrant


In [5]:
# Connect to OpenAI
openai_api = OpenAI(api_key=openai_key)

# Connect to Gemini
gemini_api = genai.Client(api_key=gemini_key)

In [6]:
# Create branch chunk collection
qdrant_client.create_collection(
    collection_name="branch_chunks",
    vector_params=models.VectorParams(
        size=3072,
        distance=models.Distance.COSINE
    )
)

[2026-02-25 14:34:25] Collection 'branch_chunks' created


In [7]:
GEMINI_MODEL = "gemini-embedding-001"
TASK_TYPE = "SEMANTIC_SIMILARITY"


In [8]:
with open("./mock.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    chunks = [j['chunk'] for j in data]

In [9]:
BATCH_SIZE = 65

In [ ]:

results_list = []
batches = []


In [11]:
for batch in range (0, len(chunks), BATCH_SIZE):
    batch = chunks[batch:batch + BATCH_SIZE]
    batches.append(batch)
    
for batch in batches:
    text_data = batch
    result = gemini_api.models.embed_content(
        model="gemini-embedding-001",
        contents=text_data,
    )
    results_list.append(result)
    
    print(f"Processed batch of size {len(batch)} waiting to respect rate limits...")
    time.sleep(2)  # To respect rate limits



In [ ]:
from uuid import UUID
import uuid
 
branch_id = []
artifact_id = []
text_chunks = []
for idx, item in enumerate(data):
    value = item["uuid"]
    branch_id.append(value)
    text_chunks.append(item["chunk"])
    artifact_id.append(value.split(".")[0])
    
    item["uuid"] = uuid.uuid5(uuid.NAMESPACE_DNS, value)


print (len(results_list), len(text_chunks), len(branch_id), len(artifact_id))
points = []
counter = 0
for result in results_list:
    for emb in result.embeddings:
        points.append(models.PointStruct(
            id=str(UUID(str(data[counter]["uuid"]))),
            vector=emb.values,
            payload={"text": text_chunks[counter], "branch_id": branch_id[counter], "artifact_id": artifact_id[counter]},
            )
        )
        print(counter)
        counter += 1

qdrant_client.client.upsert(
    collection_name="branch_chunks",
    points=points,
)

0 184 184 184


UnexpectedResponse: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Bad request: Empty update request"},"time":0.000485375}'